# Hibiki-Sw Stage 2: Audio Pretraining

**GPU Budget: ~12 hours** (80k steps across multiple sessions, 2x T4)

Train the Temporal + Depth Transformer on monolingual en+sw audio.
This teaches the model to generate audio tokens.

**Strategy for 30hr weekly limit:**
- This stage requires ~12 hours total
- Run in 2 sessions of ~6 hours each
- Use checkpoint resumption between sessions
- Save checkpoints to HF Hub for persistence

**Prerequisites:**
- Stage 1 checkpoint
- Audio token data from Notebook 00

In [ ]:
!pip install -q transformers accelerate sentencepiece pyyaml tensorboard

In [ ]:
import os
import torch

print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

REPO_DIR = "/kaggle/input/hibiki-sw-code/hibiki-sw"
DATA_DIR = "/kaggle/input/hibiki-sw-data"
OUTPUT_DIR = "/kaggle/working/stage2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Stage 1 checkpoint - adjust path based on where you stored it
STAGE1_CKPT = "/kaggle/input/hibiki-sw-stage1/checkpoint_final.pt"

# For resuming interrupted training:
RESUME_CKPT = None  # Set to checkpoint path if resuming
# RESUME_CKPT = "/kaggle/input/hibiki-sw-stage2-partial/checkpoint_40000.pt"

In [ ]:
# Merge en + sw audio token directories into a single dir
AUDIO_DIR = "/kaggle/working/audio_tokens_combined"
os.makedirs(AUDIO_DIR, exist_ok=True)

# Symlink all .npy files from both languages
import glob
for src_dir in [f"{DATA_DIR}/audio_tokens/cv_en", f"{DATA_DIR}/audio_tokens/cv_sw"]:
    for f in glob.glob(f"{src_dir}/*.npy"):
        basename = os.path.basename(f)
        dst = os.path.join(AUDIO_DIR, basename)
        if not os.path.exists(dst):
            os.symlink(f, dst)

n_files = len(glob.glob(f"{AUDIO_DIR}/*.npy"))
print(f"Combined audio tokens: {n_files} files")

In [ ]:
# Copy code to working dir
!cp -r {REPO_DIR}/* /kaggle/working/hibiki-sw/

In [ ]:
%%time
# Launch Stage 2 training
resume_flag = f"--resume {RESUME_CKPT}" if RESUME_CKPT else ""

!cd /kaggle/working/hibiki-sw && torchrun \
    --nproc_per_node=2 \
    --master_port=29500 \
    training/train_audio.py \
    --config configs/model_100m.yaml \
    --data_dir {AUDIO_DIR} \
    --stage1_ckpt {STAGE1_CKPT} \
    --output_dir {OUTPUT_DIR} \
    {resume_flag}

In [ ]:
# Check training progress & latest checkpoint
import glob
ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint_*.pt"))
print(f"Checkpoints saved: {len(ckpts)}")
for c in ckpts:
    ckpt = torch.load(c, map_location="cpu")
    print(f"  {os.path.basename(c)}: step {ckpt['step']}")

In [ ]:
# Upload latest checkpoint for cross-session persistence
# Uncomment and set your credentials:
# from huggingface_hub import HfApi
# api = HfApi(token="YOUR_TOKEN")
# latest = ckpts[-1] if ckpts else f"{OUTPUT_DIR}/checkpoint_final.pt"
# api.create_repo("YOUR_USER/hibiki-sw-stage2", exist_ok=True, private=True)
# api.upload_file(
#     path_or_fileobj=latest,
#     path_in_repo=os.path.basename(latest),
#     repo_id="YOUR_USER/hibiki-sw-stage2",
# )
print("Stage 2 session complete. Upload checkpoint for next session.")